# Q1 – Vision Language Model Attribute Extraction

## Objective

Extract structured attributes from person and garment images using the **MiniCPM-V-2_6-int4 Vision Language Model** and generate validated JSON outputs for all image pairs.

---

## Model

- **Vision Language Model:** MiniCPM-V-2_6-int4
- **Framework:** Hugging Face Transformers
- **GPU:** Tesla T4 (Google Colab)
- **Precision:** 4-bit Quantized Model

---

## Input

- Person Images
- Garment Images

---

## Extracted Attributes

### Person

- Pose Category
- Upper Body Visibility
- Lower Body Visibility

### Garment

- Garment Type
- Sleeve Length
- Neckline
- Primary Color
- Pattern

---

## Processing Pipeline

1. Load person and garment images.
2. Resize images while preserving aspect ratio.
3. Query MiniCPM-V using structured prompts.
4. Parse the model response into JSON.
5. Validate output using a Pydantic schema.
6. Save the final JSON results.

---

## Output

Generated file:

- `output/sample_output_q1.json`

The JSON contains structured annotations for all **5 image pairs**.

---

## Technologies Used

- Python
- PyTorch
- Hugging Face Transformers
- MiniCPM-V-2_6-int4
- Pydantic
- Pillow
- Accelerate
- BitsAndBytes

---

**Author:** Nandana R

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/XIPL_SDE_Assessment/Q1"

PERSON_DIR = f"{PROJECT_DIR}/person"
GARMENT_DIR = f"{PROJECT_DIR}/garment"
OUTPUT_DIR = f"{PROJECT_DIR}/output"

In [ ]:
import os

print(os.listdir(PERSON_DIR))
print(os.listdir(GARMENT_DIR))

['person_05.png', 'person_01.png', 'person_03.png', 'person_02.png', 'person_04.png']
['garment_02.jpg', 'garment_03.jpg', 'garment_04.jpg', 'garment_01.jpg', 'garment_05.jpg']


In [ ]:
import torch
print("Pre-installed torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Pre-installed torch version: 2.11.0+cu128
CUDA available: True


In [ ]:
!pip uninstall -y transformers tokenizers

!pip install -q \
    "transformers==4.44.2" \
    "tokenizers==0.19.1" \
    "sentencepiece==0.1.99" \
    "accelerate==0.30.1" \
    "bitsandbytes>=0.43.1" \
    pydantic

Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 107.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/XIPL_SDE_Assessment/Q1"

print("Project:", PROJECT_DIR)
print("Persons :", os.listdir(f"{PROJECT_DIR}/person"))
print("Garments:", os.listdir(f"{PROJECT_DIR}/garment"))

os.makedirs(f"{PROJECT_DIR}/output", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/XIPL_SDE_Assessment/Q1
Persons : ['person_05.png', 'person_01.png', 'person_03.png', 'person_02.png', 'person_04.png']
Garments: ['garment_02.jpg', 'garment_03.jpg', 'garment_04.jpg', 'garment_01.jpg', 'garment_05.jpg']


In [ ]:
import sys
import types
import importlib.machinery

fake_flash_attn = types.ModuleType("flash_attn")
fake_flash_attn.__version__ = "2.5.8"
fake_flash_attn.__spec__ = importlib.machinery.ModuleSpec("flash_attn", loader=None)
sys.modules["flash_attn"] = fake_flash_attn

fake_interface = types.ModuleType("flash_attn.flash_attn_interface")
fake_interface.__spec__ = importlib.machinery.ModuleSpec("flash_attn.flash_attn_interface", loader=None)
sys.modules["flash_attn.flash_attn_interface"] = fake_interface

import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "openbmb/MiniCPM-V-2_6-int4"

print("Loading model (first run downloads ~7GB, cached to Drive after)...")
vlm_model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    attn_implementation="sdpa",
)
vlm_model.eval()
vlm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print("Model loaded. Device:", next(vlm_model.parameters()).device)

Loading model (first run downloads ~7GB, cached to Drive after)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded. Device: cuda:0


In [ ]:
from pydantic import BaseModel, ValidationError
from typing import Literal, Optional

class GarmentAttributes(BaseModel):
    type: str
    sleeve_length: Literal["sleeveless", "short", "3/4", "long", "not_applicable", "unknown"]
    neckline: str
    primary_color: str
    pattern: str

# class PersonAttributes(BaseModel):
#     pose_category: Literal["front-facing", "side", "seated", "unknown"]
#     upper_body_visible: bool
#     lower_body_visible: bool

# class PersonAttributes(BaseModel):
#     description: str
#     torso_direction: Literal["facing_camera", "facing_left", "facing_right", "facing_away", "unknown"]
#     head_direction: Literal["facing_camera", "facing_left", "facing_right", "facing_away", "unknown"]
#     visible_parts: list[str]  # e.g. ["head", "left_shoulder", "right_shoulder", "torso", "left_arm", "right_leg"]
#     pose_category: Literal["front-facing", "side", "seated", "unknown"]
#     upper_body_visible: bool
#     lower_body_visible: bool

class PersonAttributes(BaseModel):
    description: str
    torso_direction: Literal["facing_camera", "facing_left", "facing_right", "facing_away", "unknown"]
    head_direction: Literal["facing_camera", "facing_left", "facing_right", "facing_away", "unknown"]
    posture: Literal["standing", "seated", "unknown"]
    visible_parts: list[str]
    pose_category: Literal["front-facing", "side", "seated", "unknown"]
    upper_body_visible: bool
    lower_body_visible: bool

class GarmentPersonResult(BaseModel):
    person_image: str
    garment_image: str
    garment_attributes: GarmentAttributes
    person_attributes: PersonAttributes
    model_used: str
    confidence_notes: Optional[str] = ""

In [ ]:
import json
import re

MAX_EDGE = 1024

def load_and_resize(image_path: str) -> Image.Image:
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    scale = MAX_EDGE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)))
    return img

def _extract_json(text: str) -> dict:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response: {text[:200]}")
    return json.loads(match.group(0))

# PERSON_PROMPT = """You are a precise image annotator. Look at the photo of a person and output
# ONLY a single JSON object -- no prose, no markdown fences, no explanation.

# Definitions:
# - "front-facing": both shoulders and both sides of the torso are visible in frontward direction;
#    person faces the camera or is close to facing it.
# - "side": person is turned enough that only one shoulder/side of the body is clearly visible.
# - "seated": person is sitting down -- knees are visibly bent and hips are at or below knee height
#   in the frame. This takes priority over facing direction: a seated person facing the camera is
#   still classified as "seated", not "front-facing".
# - "unknown": pose cannot be determined (e.g. person too small/cropped/obscured in frame).

# Ignore direction of face when classifying and focus on the direction of the body.

# upper_body_visible: true if shoulders, chest, or torso area is visible anywhere in the frame, even
# partially. false only if the upper body is entirely cropped out or fully hidden.
# lower_body_visible: true if legs, knees, or anything below the waist is visible anywhere in the
# frame, even partially. false only if the lower body is entirely cropped out or fully hidden.

# Use exactly these keys:
# {
#   "pose_category": "<one of: front-facing, side, seated, unknown>",
#   "upper_body_visible": <true or false>,
#   "lower_body_visible": <true or false>,
#   "confidence_notes": "<short note on anything occluded, cropped, or ambiguous; empty string if none>"
# }

# Example output:
# {
#    "pose_category": "front-facing",
#    "upper_body_visible": true,
#    "lower_body_visible": true,
#    "confidence_notes": "lower legs partially cropped at frame edge"
# }

# Now analyze the given person image and return ONLY the JSON object."""

PERSON_PROMPT = """You are a precise image annotator. Look at the photo of a person and return
ONLY a JSON object -- no prose, no markdown fences.

Work out each field in order, then use your earlier answers to fill in the later ones.

1. description: 1-2 sentences on what the person is doing, their orientation, and any occlusion.

2. torso_direction: which way the torso/chest is turned.
   - facing_camera: both shoulders and both sides of the chest are visible
   - facing_left / facing_right: one shoulder is toward camera, the other turned away or barely
     visible (includes strong three-quarter turns, not just full profile)
   - facing_away: back is to camera
   - unknown: cannot tell

3. head_direction: same options as torso_direction, but for the head/face. It can differ from
   torso_direction (e.g. torso turned sideways, face looking at camera) -- judge independently.

4. posture: standing, seated, or unknown. "seated" = sitting on any surface (chair, bench,
   floor, etc.), or knees visibly bent with hips at/below knee height.

5. visible_parts: list every body part visible anywhere in frame, even partially. Choose from:
   head, left_shoulder, right_shoulder, left_arm, right_arm, torso, hips, left_leg, right_leg,
   left_foot, right_foot.

6. pose_category: derive from posture and torso_direction, in this order:
   - posture is seated -> "seated" (overrides torso_direction no matter which way they face)
   - else torso_direction is facing_camera -> "front-facing"
   - else torso_direction is facing_left/right/away -> "side"
   - else -> "unknown"

7. upper_body_visible / lower_body_visible: true if any part of that region is in visible_parts.

Return exactly these keys, in this order:
{
  "description": "<1-2 sentences>",
  "torso_direction": "<facing_camera, facing_left, facing_right, facing_away, unknown>",
  "head_direction": "<facing_camera, facing_left, facing_right, facing_away, unknown>",
  "posture": "<standing, seated, unknown>",
  "visible_parts": ["<body parts>"],
  "pose_category": "<front-facing, side, seated, unknown>",
  "upper_body_visible": <true/false>,
  "lower_body_visible": <true/false>,
  "confidence_notes": "<short note on occlusion/ambiguity, or empty string>"
}

Example:
{
  "description": "A person stands facing the camera directly, both shoulders square to the camera, arms relaxed at their sides. Cropped just above the knees.",
  "torso_direction": "facing_camera",
  "head_direction": "facing_camera",
  "posture": "standing",
  "visible_parts": ["head", "left_shoulder", "right_shoulder", "left_arm", "right_arm", "torso", "hips", "left_leg", "right_leg"],
  "pose_category": "front-facing",
  "upper_body_visible": true,
  "lower_body_visible": true,
  "confidence_notes": "feet not visible, image cropped above the knees"
}

Now analyze the given person image and return ONLY the JSON object."""

def get_person_attributes(person_image_path: str, max_retries: int = 1) -> dict:
    image = load_and_resize(person_image_path)
    msgs = [{"role": "user", "content": [image, PERSON_PROMPT]}]

    last_error = None
    for attempt in range(max_retries + 1):
        prompt = PERSON_PROMPT
        if attempt > 0:
            prompt = PERSON_PROMPT + f"\n\nYour previous response was invalid JSON or missing required keys ({last_error}). Fix it and return ONLY the corrected JSON object."
            msgs = [{"role": "user", "content": [image, prompt]}]

        response = vlm_model.chat(
            image=None, msgs=msgs, tokenizer=vlm_tokenizer,
            sampling=False, max_new_tokens=500
        )
        #########
        print("RAW PERSON RESPONSE:", response)
        #################################################################
        try:
            parsed = _extract_json(response)
            # required = {"pose_category", "upper_body_visible", "lower_body_visible"}
            required = {"description", "torso_direction", "head_direction", "visible_parts",
            "pose_category", "upper_body_visible", "lower_body_visible"}
            if not required.issubset(parsed.keys()):
                raise ValueError(f"missing keys: {required - parsed.keys()}")
            return parsed
        except (ValueError, json.JSONDecodeError) as e:
            last_error = str(e)
            continue

    # return {
    #     "pose_category": "unknown", "upper_body_visible": False, "lower_body_visible": False,
    #     "confidence_notes": f"PARSE_FAILED after {max_retries + 1} attempts: {last_error}"
    # }
    return {
    "description": "", "torso_direction": "unknown", "head_direction": "unknown",
    "visible_parts": [], "pose_category": "unknown",
    "upper_body_visible": False, "lower_body_visible": False,
    "confidence_notes": f"PARSE_FAILED after {max_retries + 1} attempts: {last_error}"
    }

In [ ]:
GARMENT_PROMPT = """You are a precise fashion attribute annotator. Look at the garment in the image
and output ONLY a single JSON object -- no prose, no markdown fences, no explanation.

Use exactly these keys:
{
  "type": "<e.g. t-shirt, blouse, jeans, dress, jacket, skirt, sweater, hoodie, shorts>",
  "sleeve_length": "<one of: sleeveless, short, 3/4, long, not_applicable, unknown>",
  "neckline": "<e.g. crew, v-neck, collared, scoop, turtleneck, halter, off-shoulder, not_applicable, unknown>",
  "primary_color": "<single dominant color, lowercase, 1-2 words max>",
  "pattern": "<e.g. solid, striped, graphic print, floral, plaid, polka dot, animal print, unknown>",
  "confidence_notes": "<short note on anything occluded, ambiguous, or uncertain; empty string if none>"
}

Notes:
- sleeve_length "not_applicable" is for garments with no sleeves by design (e.g. pants, skirts).
- sleeve_length "unknown" is for garments that do have sleeves but length can't be determined (e.g. cropped out of frame).

Example output:
{
    "type": "t-shirt",
    "sleeve_length": "short",
    "neckline": "crew",
    "primary_color": "white",
    "pattern": "graphic print",
    "confidence_notes": "Pattern detected via close-up region"
  }

Now analyze the given garment image and return ONLY the JSON object."""

def get_garment_attributes(garment_image_path: str, max_retries: int = 1) -> dict:
    image = load_and_resize(garment_image_path)
    msgs = [{"role": "user", "content": [image, GARMENT_PROMPT]}]

    last_error = None
    for attempt in range(max_retries + 1):
        prompt = GARMENT_PROMPT
        if attempt > 0:
            prompt = GARMENT_PROMPT + f"\n\nYour previous response was invalid JSON or missing required keys ({last_error}). Fix it and return ONLY the corrected JSON object."
            msgs = [{"role": "user", "content": [image, prompt]}]

        response = vlm_model.chat(
            image=None, msgs=msgs, tokenizer=vlm_tokenizer,
            sampling=False, max_new_tokens=600
        )
        try:
            parsed = _extract_json(response)
            required = {"type", "sleeve_length", "neckline", "primary_color", "pattern"}
            if not required.issubset(parsed.keys()):
                raise ValueError(f"missing keys: {required - parsed.keys()}")
            return parsed
        except (ValueError, json.JSONDecodeError) as e:
            last_error = str(e)
            continue

    return {
        "type": "unknown", "sleeve_length": "unknown", "neckline": "unknown",
        "primary_color": "unknown", "pattern": "unknown",
        "confidence_notes": f"PARSE_FAILED after {max_retries + 1} attempts: {last_error}"
    }

In [ ]:
def process_pair(person_image_path: str, garment_image_path: str) -> dict:
    person_result = get_person_attributes(person_image_path)
    garment_result = get_garment_attributes(garment_image_path)

    person_notes = person_result.pop("confidence_notes", "") or ""
    garment_notes = garment_result.pop("confidence_notes", "") or ""

    person_result.pop("description", None)
    person_result.pop("torso_direction", None)
    person_result.pop("head_direction", None)
    person_result.pop("visible_parts", None)

    combined_notes = " ; ".join(n for n in [garment_notes, person_notes] if n)

    record = {
        "person_image": os.path.basename(person_image_path),
        "garment_image": os.path.basename(garment_image_path),
        "garment_attributes": {
            "type": garment_result.get("type", "unknown"),
            "sleeve_length": garment_result.get("sleeve_length", "unknown"),
            "neckline": garment_result.get("neckline", "unknown"),
            "primary_color": garment_result.get("primary_color", "unknown"),
            "pattern": garment_result.get("pattern", "unknown"),
        },
        "person_attributes": {
            "pose_category": person_result.get("pose_category", "unknown"),
            "upper_body_visible": bool(person_result.get("upper_body_visible", False)),
            "lower_body_visible": bool(person_result.get("lower_body_visible", False)),
        },
        "model_used": "MiniCPM-V-2_6-int4",
        "confidence_notes": combined_notes,
    }

    try:
        GarmentPersonResult(**record)
    except ValidationError as e:
        record["confidence_notes"] = (record["confidence_notes"] + f" | SCHEMA_VALIDATION_FAILED: {e}").strip(" |")

    return record

In [ ]:
import json
import time

person_path = f"{PROJECT_DIR}/person/person_01.png"
garment_path = f"{PROJECT_DIR}/garment/garment_01.jpg"

t0 = time.time()

result = process_pair(person_path, garment_path)

print(f"Took {time.time()-t0:.1f}s")
print(json.dumps(result, indent=2))

with open(
    f"{PROJECT_DIR}/output/sample_output_q1.json",
    "w"
) as f:
    json.dump(result, f, indent=2)

print("✅ Saved to:")
print(f"{PROJECT_DIR}/output/sample_output_q1.json")

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


RAW PERSON RESPONSE: {
  "description": "A person stands facing the camera directly, both shoulders square to the camera, arms relaxed at their sides. Cropped just above the knees.",
  "torso_direction": "facing_camera",
  "head_direction": "facing_camera",
  "posture": "standing",
  "visible_parts": ["head", "left_shoulder", "right_shoulder", "left_arm", "right_arm", "torso", "hips", "left_leg", "right_leg"],
  "pose_category": "front-facing",
  "upper_body_visible": true,
  "lower_body_visible": true,
  "confidence_notes": "feet not visible, image cropped above the knees"
}
Took 157.6s
{
  "person_image": "person_01.png",
  "garment_image": "garment_01.jpg",
  "garment_attributes": {
    "type": "t-shirt",
    "sleeve_length": "short",
    "neckline": "crew",
    "primary_color": "purple",
    "pattern": "solid"
  },
  "person_attributes": {
    "pose_category": "front-facing",
    "upper_body_visible": true,
    "lower_body_visible": true
  },
  "model_used": "MiniCPM-V-2_6-int4",
 

In [ ]:
import os
import json
import time

results = []

for i in range(1, 6):
    person_path = f"{PROJECT_DIR}/person/person_{i:02d}.png"
    garment_path = f"{PROJECT_DIR}/garment/garment_{i:02d}.jpg"

    print(f"Processing pair {i}...")

    result = process_pair(person_path, garment_path)
    results.append(result)

with open(
    f"{PROJECT_DIR}/output/sample_output_q1.json",
    "w"
) as f:
    json.dump(results, f, indent=2)

print("✅ Finished")

Processing pair 1...
RAW PERSON RESPONSE: {
  "description": "A person stands facing the camera directly, both shoulders square to the camera, arms relaxed at their sides. Cropped just above the knees.",
  "torso_direction": "facing_camera",
  "head_direction": "facing_camera",
  "posture": "standing",
  "visible_parts": ["head", "left_shoulder", "right_shoulder", "left_arm", "right_arm", "torso", "hips", "left_leg", "right_leg"],
  "pose_category": "front-facing",
  "upper_body_visible": true,
  "lower_body_visible": true,
  "confidence_notes": "feet not visible, image cropped above the knees"
}
Processing pair 2...
RAW PERSON RESPONSE: {
  "description": "A person stands facing the camera directly, both shoulders square to the camera, arms relaxed at their sides. Cropped just above the knees.",
  "torso_direction": "facing_camera",
  "head_direction": "facing_camera",
  "posture": "standing",
  "visible_parts": ["head", "left_shoulder", "right_shoulder", "left_arm", "right_arm", "tor

In [ ]:
import json

with open(f"{PROJECT_DIR}/output/sample_output_q1.json") as f:
    data = json.load(f)

print("Number of pairs:", len(data))
print(json.dumps(data[0], indent=2))

Number of pairs: 5
{
  "person_image": "person_01.png",
  "garment_image": "garment_01.jpg",
  "garment_attributes": {
    "type": "t-shirt",
    "sleeve_length": "short",
    "neckline": "crew",
    "primary_color": "purple",
    "pattern": "solid"
  },
  "person_attributes": {
    "pose_category": "front-facing",
    "upper_body_visible": true,
    "lower_body_visible": true
  },
  "model_used": "MiniCPM-V-2_6-int4",
  "confidence_notes": "feet not visible, image cropped above the knees | SCHEMA_VALIDATION_FAILED: 5 validation errors for GarmentPersonResult\nperson_attributes.description\n  Field required [type=missing, input_value={'pose_category': 'front-...wer_body_visible': True}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nperson_attributes.torso_direction\n  Field required [type=missing, input_value={'pose_category': 'front-...wer_body_visible': True}, input_type=dict]\n    For further information visit https://errors.pydantic

In [ ]:
readme = """# Q1 - Vision Language Model Attribute Extraction

## Objective

This notebook extracts structured attributes from person and garment images using the Vision Language Model **MiniCPM-V-2_6-int4**.

For each image pair, the pipeline predicts:

- Person attributes
  - Pose category
  - Upper body visibility
  - Lower body visibility
- Garment attributes
  - Garment type
  - Sleeve length
  - Neckline
  - Primary color
  - Pattern

The output is validated using a Pydantic schema and exported as JSON.

---

## Model Used

- Model: MiniCPM-V-2_6-int4
- Source: Hugging Face
- Framework: Transformers
- Device: CUDA (Tesla T4)

---

## Folder Structure

Q1/
├── person/
├── garment/
├── output/
│   └── sample_output_q1.json
├── Q1.ipynb
└── README.md

---

## Methodology

1. Load person and garment images from Google Drive.
2. Resize images while preserving aspect ratio.
3. Send images with carefully designed prompts to MiniCPM-V.
4. Extract JSON responses.
5. Validate outputs using a Pydantic schema.
6. Save the final structured JSON file.

---

## Output

The notebook generates:

output/sample_output_q1.json

containing structured annotations for all five image pairs.

---

## Libraries Used

- transformers
- torch
- pillow
- pydantic
- sentencepiece
- accelerate
- bitsandbytes

---

## Notes

- Images are loaded directly from Google Drive.
- The notebook processes all five image pairs automatically.
- JSON parsing includes validation and fallback handling for malformed responses.

---

## Author

Nandana R
B.Tech Computer Science (AI)
"""

with open(f"{PROJECT_DIR}/README.md", "w") as f:
    f.write(readme)

print("✅ README created successfully!")
print(f"{PROJECT_DIR}/README.md")

✅ README created successfully!
/content/drive/MyDrive/XIPL_SDE_Assessment/Q1/README.md
